## Thresholding (4 equal ranges → 0,1,2,3)

In [1]:
%%writefile q6.cu
#include <cuda_runtime.h>
#include <stdio.h>
#include <stdlib.h>
#include <time.h>

#define N 60
#define LOW 10
#define HIGH 99
#define BLOCK_SIZE 8

// ── CUDA Kernel
// Range = 99-10 = 89, divided into 4 equal parts (~22 each)
// [10..31]=0, [32..53]=1, [54..75]=2, [76..99]=3
__global__ void threshKernel(int *dA, int *dC, int n)
{
    int i = blockIdx.x * blockDim.x + threadIdx.x;
    if (i < n) {
        int range = (HIGH - LOW + 1) / 4;   // = 22
        dC[i] = (dA[i] - LOW) / range;
        if (dC[i] > 3) dC[i] = 3;           // clamp top bucket
    }
}

// ── Host Function
__host__ void thresh(int *hA, int *hC, int n)
{
    int *dA, *dC;
    int size = n * sizeof(int);
    cudaMalloc((void**)&dA, size);
    cudaMalloc((void**)&dC, size);
    cudaMemcpy(dA, hA, size, cudaMemcpyHostToDevice);
    dim3 DimBlock(BLOCK_SIZE, 1, 1);
    dim3 DimGrid((int)ceil((float)n / BLOCK_SIZE), 1, 1);
    threshKernel<<<DimGrid, DimBlock>>>(dA, dC, n);
    cudaMemcpy(hC, dC, size, cudaMemcpyDeviceToHost);
    cudaFree(dA);
    cudaFree(dC);
}

// ── Main
int main()
{
    int *hA, *hC;
    hA = (int*) malloc(N * sizeof(int));
    hC = (int*) malloc(N * sizeof(int));

    srand(time(NULL));
    for (int i = 0; i < N; i++)
        hA[i] = rand() % (HIGH - LOW + 1) + LOW;

    printf("hA:\n");
    for (int i = 0; i < N; i++) printf("%4d", hA[i]);
    printf("\n");

    thresh(hA, hC, N);

    printf("\nhC (Thresholded 0-3):\n");
    for (int i = 0; i < N; i++) printf("%4d", hC[i]);
    printf("\n");

    free(hA); free(hC);
    return 0;
}

Writing q6.cu


In [2]:
!nvcc -arch=sm_75 q6.cu -o q6

!./q6

hA:
  65  87  27  13  53  34  43  96  91  72  19  61  37  26  27  34  42  47  87  54  16  33  36  77  40  51  98  34  48  71  30  65  20  99  68  63  33  11  59  24  35  30  37  62  98  16  86  41  54  36  47  22  59  73  51  52  24  49  38  62

hC (Thresholded 0-3):
   2   3   0   0   1   1   1   3   3   2   0   2   1   0   0   1   1   1   3   2   0   1   1   3   1   1   3   1   1   2   0   2   0   3   2   2   1   0   2   0   1   0   1   2   3   0   3   1   2   1   1   0   2   2   1   1   0   1   1   2
